<a href="https://colab.research.google.com/github/AjayKumar22633/GRU_DeepLearning/blob/main/GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/general_knowledge_qa.csv')

In [ ]:
#tokenize
def tokenize(text):
  test = text.lower()
  text = text.replace('?', '')
  text = text.replace("'","")
  return text.split()

In [ ]:
tokenize('Now, Hal, what time of day is it, lad?')


['Now,', 'Hal,', 'what', 'time', 'of', 'day', 'is', 'it,', 'lad']

In [ ]:
# Initialize vocabulary with UNK token
vocab = {'<UNK>': 0}

# Redefine build_vocab to add tokens to the dictionary
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    for token in tokenized_question + tokenized_answer:
        if token not in vocab:
            vocab[token] = len(vocab)

# Populate the vocabulary using all rows of the dataframe
df.apply(build_vocab, axis=1)
print(f"Vocabulary built successfully! Total unique tokens: {len(vocab)}")

Vocabulary built successfully! Total unique tokens: 2573


In [ ]:
def build_vocab(row):
 print(row['question'], row['answer'])
 tokenized_Player = tokenize(row['question'])
 tokenized_PlayerLine = tokenize(row['answer'])

 merged_tokens = tokenized_Player + tokenized_PlayerLine
 for token in merged_tokens:
  if token not in vocab:
    vocab[token] = len(vocab)

In [ ]:
import pandas as pd

# Redefine tokenize to safely handle non-string (NaN/float) inputs
def tokenize(text):
    text_str = str(text) if not pd.isna(text) else ""
    test = text_str.lower()
    text_str = text_str.replace('?', '')
    text_str = text_str.replace("'", "")
    return text_str.split()


def build_vocab(row):
    tokenized_Player = tokenize(row['question'])
    tokenized_PlayerLine = tokenize(row['answer'])
    # To avoid printing 100k lines, let's only print the first few non-empty ones
    if len(tokenized_Player) > 0 and len(tokenized_PlayerLine) > 0:
        print(tokenized_Player, tokenized_PlayerLine)

if isinstance(df, str):
    df = pd.read_csv(df)

# Run the apply function (limiting to the first 50 rows as a test to avoid huge output)
df.head().apply(build_vocab, axis=1)

['How', 'many', 'days', 'do', 'we', 'have', 'in', 'a', 'week'] ['Seven']
['How', 'many', 'days', 'are', 'there', 'in', 'a', 'normal', 'year'] ['365', '(not', 'a', 'leap', 'year)']
['How', 'many', 'colors', 'are', 'there', 'in', 'a', 'rainbow'] ['7']
['Which', 'animal', 'is', 'known', 'as', 'the', '‘Ship', 'of', 'the', 'Desert’'] ['Camel']
['How', 'many', 'letters', 'are', 'there', 'in', 'the', 'English', 'alphabet'] ['26']


,0
0,None
1,None
2,None
3,None
4,None


In [ ]:
#convert words to numerical indices
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
class QADatase(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab
  def __len__(self):
    return self.df.shape[0]
  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)
    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [ ]:
# Re-create dataset and dataloader using the updated vocabulary
dataset = QADatase(df, vocab)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
dataset[10]

(tensor([ 1,  2, 57, 19, 11, 12]), tensor([40, 58, 59, 60]))

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[  62,   24, 1708,  350,  155, 1709,  949]]) tensor([1710, 1711])
tensor([[ 22, 210,  24,  25,  26,  27, 211,  29,  27,  70]]) tensor([212])
tensor([[   1,    2,   38,   95,   27, 2222,   29,    8, 2223,    6]]) tensor([653])
tensor([[ 462,  799,  800, 1268,  903,   24,   25, 1269]]) tensor([602])
tensor([[  22,   29, 1234,   24,    8, 1235, 1236]]) tensor([ 736, 1237])
tensor([[  22,   24,   27, 1452,  164, 1453,   29,  312]]) tensor([1454])
tensor([[  62,   95, 1306,   79,   29,    8, 1198, 1304, 1305]]) tensor([1307])
tensor([[  62,   24, 1654,   14]]) tensor([ 912,   24,   27, 2532,   29, 2019])
tensor([[571, 572,  24, 573,  26, 574, 575, 452, 285, 453]]) tensor([454])
tensor([[2244, 2245,   24,    7,   93, 2245,   29,   27,  221]]) tensor([2085])
tensor([[ 310, 1226, 1387, 1388,   24, 1389,  226,   93,  972]]) tensor([949])
tensor([[  62,   24,   27,  350,   29, 2290, 2291, 2292, 2293,  279,  724,  435,
          137,  436, 2294]]) tensor([2295])
tensor([[1985,   93,  236,

In [ ]:
import torch.nn as nn

In [ ]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size, emb=128, hid=256):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, emb, padding_idx=0)
    self.enc = nn.GRU(emb, hid, batch_first=True)
    self.dec = nn.GRU(emb, hid, batch_first=True)
    self.fc = nn.Linear(hid, vocab_size)


  def forward(self, question, a_in):
    _, h = self.enc(self.embedding(question))
    out, _ = self.dec(self.embedding(a_in), h)
    return self.fc(out)

In [ ]:
learning_rate = 0.001
epochs = 20

In [ ]:
# Re-instantiate the model with the true vocabulary size
model = SimpleRNN(len(vocab))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Re-initialize the model and optimizer to apply the class definition changes
model = SimpleRNN(len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# training loop
for epoch in range(epochs):
  total_loss = 0
  for question, answer in dataloader:
    # Skip empty sequences if they occur
    if question.size(1) == 0 or answer.size(1) == 0:
      continue

    optimizer.zero_grad()

    # Pass the question and target answer to the corrected model
    output = model(question, answer)

    # Reshape to align [Batch * Seq_Len, Vocab_Size] with target [Batch * Seq_Len]
    loss = criterion(output.contiguous().view(-1, len(vocab)), answer.contiguous().view(-1))

    # gradients
    loss.backward()
    # update
    optimizer.step()
    total_loss = total_loss + loss.item()
  if (epoch + 1) % 5 == 0 or epoch == 0:
    print(f"Epoch: {epoch+1:02d}, Loss: {total_loss:4f}")

Epoch: 01, Loss: 6005.072963
Epoch: 05, Loss: 10.785865
Epoch: 10, Loss: 0.904801
Epoch: 15, Loss: 0.085067
Epoch: 20, Loss: 0.007862


In [ ]:
def predict(model, question, max_len=5, threshold=0.10):
  # Convert question to numbers
  numerical_question = text_to_indices(question, vocab)
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  predicted_tokens = []

  # We will generate words one by one (auto-regressively)
  for step in range(max_len):
    # Prepare the decoder input sequence 'a_in'
    # It starts with a dummy start token (0), followed by numerical indices of words predicted so far
    current_decoder_input = [0] + [vocab[t] for t in predicted_tokens if t in vocab]
    decoder_tensor = torch.tensor(current_decoder_input).unsqueeze(0) # shape: [1, seq_len]

    # Run forward pass
    with torch.no_grad():
      output = model(question_tensor, decoder_tensor)

    # Get probabilities for the last predicted step
    probs = torch.nn.functional.softmax(output, dim=-1) # shape: [1, seq_len, Vocab_Size]
    value, index = torch.max(probs[0, -1, :], dim=-1)

    prob_val = value.item()
    idx_val = index.item()

    vocab_keys = list(vocab.keys())
    next_word = vocab_keys[idx_val]

    # Stop generating if the model becomes unconfident
    if prob_val < threshold:
      break

    predicted_tokens.append(next_word)

  if len(predicted_tokens) == 0:
    print("I don't know")
  else:
    print(" ".join(predicted_tokens))

In [ ]:
def inspect_predict(model, question):
  # Convert question to numbers
  numerical_question = text_to_indices(question, vocab)
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # Since SimpleRNN.forward expects both encoder inputs (question) and decoder inputs (a_in),
  # we pass a starting/dummy input tensor of zeros for the decoder sequence.
  decoder_input = torch.zeros((1, 1), dtype=torch.long)

  # Send to model
  output = model(question_tensor, decoder_input)
  probs = torch.nn.functional.softmax(output, dim=-1)

  # Get the top 5 predictions and their probabilities
  top_probs, top_indices = torch.topk(probs, k=5, dim=-1)

  print(f"Question: {question}")
  print("\nTop 5 predictions:")
  vocab_keys = list(vocab.keys())
  for i in range(5):
    prob = top_probs[0, 0, i].item()
    idx = top_indices[0, 0, i].item()
    word = vocab_keys[idx]
    print(f"  {word}: {prob:.4f} ({prob*100:.2f}%)")

# Run inspection on the question
inspect_predict(model, "How many days are there in a normal year?")

Question: How many days are there in a normal year?

Top 5 predictions:
  365: 0.0821 (8.21%)
  11: 0.0240 (2.40%)
  1916: 0.0177 (1.77%)
  28: 0.0164 (1.64%)
  November: 0.0145 (1.45%)


In [ ]:
predict(model, "How many days are there in a normal year?")

I don't know
